# 宠物领养速度预测 — 数据分析与建模

基于马来西亚 PetFinder 平台数据，探索影响宠物被领养速度的关键因素，并构建预测模型。


## 1. 环境准备 & 数据加载


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score)

# 画图设置
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

%matplotlib inline


In [ ]:
# 加载数据
df = pd.read_csv("../data/raw/train.csv")

# 加载标签表
breed_labels = pd.read_csv("../data/raw/BreedLabels.csv")
color_labels = pd.read_csv("../data/raw/ColorLabels.csv")
state_labels = pd.read_csv("../data/raw/state_labels.csv")

print(f"数据维度: {df.shape}")
print(f"列数: {df.shape[1]}")
df.head()


In [ ]:
# 数据类型与缺失值概览
print("=== 数据类型 ===")
print(df.dtypes)
print(f"\n=== 缺失值统计 ===")
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing if len(missing) > 0 else "无缺失值")


## 2. 目标变量分布 (AdoptionSpeed)

- 0 = 当天被领养
- 1 = 1–7 天内被领养
- 2 = 8–30 天内被领养
- 3 = 31–90 天内被领养
- 4 = 未被领养


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 柱状图
speed_counts = df['AdoptionSpeed'].value_counts().sort_index()
labels = ['当天(0)', '1-7天(1)', '8-30天(2)', '31-90天(3)', '未领养(4)']
axes[0].bar(labels, speed_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('AdoptionSpeed 各类别数量', fontsize=14)
axes[0].set_ylabel('数量')
for i, v in enumerate(speed_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=10)

# 饼图
axes[1].pie(speed_counts.values, labels=labels, autopct='%1.1f%%',
            colors=sns.color_palette("Blues", 5), startangle=90)
axes[1].set_title('AdoptionSpeed 各类别占比', fontsize=14)

plt.tight_layout()
plt.show()


## 3. 数值特征探索


In [ ]:
# 数值型特征分布
num_cols = ['Age', 'Quantity', 'Fee', 'VideoAmt', 'PhotoAmt']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    df[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} 分布', fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('频数')

# 隐藏多余的子图
axes[5].axis('off')
plt.tight_layout()
plt.show()

# 描述统计
df[num_cols].describe().T


## 4. 类别特征探索


In [ ]:
# 宠物类型分布 (1=狗, 2=猫)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Type
type_map = {1: '狗 (Dog)', 2: '猫 (Cat)'}
df['TypeName'] = df['Type'].map(type_map)
type_counts = df['TypeName'].value_counts()
axes[0, 0].bar(type_counts.index, type_counts.values, color=['#FF8C00', '#9370DB'], edgecolor='white')
axes[0, 0].set_title('宠物类型分布', fontsize=14)
for i, v in enumerate(type_counts.values):
    axes[0, 0].text(i, v + 100, str(v), ha='center', fontsize=11)

# Gender
gender_map = {1: '公 (Male)', 2: '母 (Female)', 3: '混合 (Mixed)'}
df['GenderName'] = df['Gender'].map(gender_map)
gender_counts = df['GenderName'].value_counts()
axes[0, 1].bar(gender_counts.index, gender_counts.values, color='lightcoral', edgecolor='white')
axes[0, 1].set_title('性别分布', fontsize=14)
for i, v in enumerate(gender_counts.values):
    axes[0, 1].text(i, v + 100, str(v), ha='center', fontsize=11)

# MaturitySize
size_map = {1: '小型', 2: '中型', 3: '大型', 4: '超大型'}
df['SizeName'] = df['MaturitySize'].map(size_map)
size_counts = df['SizeName'].value_counts()
axes[1, 0].bar(size_counts.index, size_counts.values, color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('体型分布', fontsize=14)

# FurLength
fur_map = {1: '短毛', 2: '中毛', 3: '长毛'}
df['FurName'] = df['FurLength'].map(fur_map)
fur_counts = df['FurName'].value_counts()
axes[1, 1].bar(fur_counts.index, fur_counts.values, color='goldenrod', edgecolor='white')
axes[1, 1].set_title('毛发长度分布', fontsize=14)

plt.tight_layout()
plt.show()


In [ ]:
# 不同类型宠物在各领养速度下的分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (pet_type, pet_name) in enumerate([(1, '狗 (Dog)'), (2, '猫 (Cat)')]):
    subset = df[df['Type'] == pet_type]
    speed_dist = subset['AdoptionSpeed'].value_counts().sort_index()
    axes[idx].bar(labels, speed_dist.values, color='steelblue', edgecolor='white')
    axes[idx].set_title(f'{pet_name} — 领养速度分布', fontsize=13)
    axes[idx].set_xlabel('AdoptionSpeed')
    axes[idx].set_ylabel('数量')

plt.tight_layout()
plt.show()


## 5. 数据预处理


In [ ]:
# 特征工程：创建派生特征
df['IsSenior'] = (df['Age'] >= 24).astype(int)     # 年龄 >= 24 个月视为老年
df['HasVideo'] = (df['VideoAmt'] > 0).astype(int)  # 是否有视频
df['HasDescription'] = df['Description'].notna().astype(int)  # 是否有文字描述
df['PhotoAmt_log'] = np.log1p(df['PhotoAmt'])      # 对数变换减少偏态

print("派生特征已创建:")
print(f"IsSenior 分布: {df['IsSenior'].value_counts().to_dict()}")
print(f"HasVideo 分布: {df['HasVideo'].value_counts().to_dict()}")
print(f"HasDescription 分布: {df['HasDescription'].value_counts().to_dict()}")


In [ ]:
# 选择用于建模的特征
feature_cols = [
    'Type', 'Age', 'Breed1', 'Gender', 'Color1',
    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed',
    'Sterilized', 'Health', 'Quantity', 'Fee', 'VideoAmt',
    'PhotoAmt_log', 'IsSenior', 'HasVideo', 'HasDescription'
]

# 构建建模数据
model_df = df[feature_cols + ['AdoptionSpeed']].dropna()
X = model_df[feature_cols]
y = model_df['AdoptionSpeed']

print(f"建模数据维度: {X.shape}")
print(f"目标分布:\n{y.value_counts().sort_index()}")


In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")
print(f"\n训练集目标分布:\n{y_train.value_counts(normalize=True).sort_index().round(3)}")
print(f"\n测试集目标分布:\n{y_test.value_counts(normalize=True).sort_index().round(3)}")


## 6. Baseline: 五分类模型 (RandomForest)


In [ ]:
# 训练五分类 RandomForest
rf_5class = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_5class.fit(X_train, y_train)

y_pred_5 = rf_5class.predict(X_test)
y_proba_5 = rf_5class.predict_proba(X_test)

print("=== 五分类 Baseline 评估 ===")
print(f"准确率 (Accuracy): {accuracy_score(y_test, y_pred_5):.4f}")
print(f"\n分类报告:\n{classification_report(y_test, y_pred_5, target_names=labels)}")


In [ ]:
# 五分类混淆矩阵
cm_5 = confusion_matrix(y_test, y_pred_5)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_5, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('五分类混淆矩阵', fontsize=14)
plt.xlabel('预测类别')
plt.ylabel('真实类别')
plt.show()


## 7. 转为二分类 — "被领养" vs "未被领养"

将 AdoptionSpeed 0–3 合并为"已领养"（1），4 保持"未领养"（0），简化问题使模型更稳定。


In [ ]:
# 二分类转换
df['Adopted'] = (df['AdoptionSpeed'] < 4).astype(int)

y_binary = model_df['AdoptionSpeed'].apply(lambda x: 1 if x < 4 else 0)
y_binary = df.loc[X.index, 'Adopted']

print("=== 二分类目标分布 ===")
print(f"未领养 (0): {(y_binary == 0).sum()} ({(y_binary == 0).mean():.1%})")
print(f"已领养 (1): {(y_binary == 1).sum()} ({(y_binary == 1).mean():.1%})")

# 可视化
fig, ax = plt.subplots(figsize=(6, 4))
counts = y_binary.value_counts()
colors = ['tomato', 'steelblue']
ax.bar(['未领养 (0)', '已领养 (1)'], [counts.get(0, 0), counts.get(1, 0)], color=colors, edgecolor='white')
ax.set_title('二分类目标分布', fontsize=14)
for i, v in enumerate([counts.get(0, 0), counts.get(1, 0)]):
    ax.text(i, v + 100, f'{v}\n({v/len(y_binary):.1%})', ha='center', fontsize=11)
plt.show()


In [ ]:
# 二分类数据划分
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# 训练二分类 RandomForest
rf_binary = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_binary.fit(X_train_b, y_train_b)

y_pred_b = rf_binary.predict(X_test_b)
y_proba_b = rf_binary.predict_proba(X_test_b)[:, 1]

print("=== 二分类模型评估 ===")
print(f"准确率 (Accuracy): {accuracy_score(y_test_b, y_pred_b):.4f}")
print(f"\n分类报告:\n{classification_report(y_test_b, y_pred_b, target_names=['未领养', '已领养'])}")


## 8. ROC-AUC — 验证模型区分能力


In [ ]:
# 计算 ROC 曲线
fpr, tpr, thresholds = roc_curve(y_test_b, y_proba_b)
auc_score = roc_auc_score(y_test_b, y_proba_b)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='随机猜测 (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.2, color='darkorange')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('假阳性率 (False Positive Rate)', fontsize=12)
plt.ylabel('真阳性率 (True Positive Rate)', fontsize=12)
plt.title('ROC 曲线 — 二分类领养预测模型', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.show()

print(f"AUC Score: {auc_score:.4f}")
print(f"\nAUC 解读: {'优秀 (>0.9)' if auc_score > 0.9 else '良好 (0.8-0.9)' if auc_score > 0.8 else '一般 (0.7-0.8)' if auc_score > 0.7 else '较差 (<0.7)'}")
print("结论: 模型具有区分被领养与未被领养宠物的能力。" if auc_score > 0.7 else "模型区分能力需进一步提升。")


## 9. 特征重要性分析


In [ ]:
# 提取特征重要性
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_binary.feature_importances_
}).sort_values('importance', ascending=False)

# 特征名映射（中英文对照）
feature_name_map = {
    'Age': '年龄', 'Breed1': '主要品种', 'Gender': '性别', 'Color1': '主要颜色',
    'MaturitySize': '体型', 'FurLength': '毛发长度', 'Type': '宠物类型',
    'Vaccinated': '接种疫苗', 'Dewormed': '驱虫', 'Sterilized': '绝育',
    'Health': '健康状况', 'Quantity': '数量', 'Fee': '领养费用',
    'VideoAmt': '视频数量', 'PhotoAmt_log': '照片数量(log)',
    'IsSenior': '是否老年', 'HasVideo': '有视频', 'HasDescription': '有描述'
}
feature_importance['feature_cn'] = feature_importance['feature'].map(feature_name_map)

# 可视化
plt.figure(figsize=(10, 8))
top_n = 15
top_features = feature_importance.head(top_n)

colors = plt.cm.Blues(np.linspace(0.4, 0.9, top_n))
bars = plt.barh(range(top_n), top_features['importance'].values[::-1], color=colors[::-1], edgecolor='white')
plt.yticks(range(top_n), top_features['feature_cn'].values[::-1])
plt.xlabel('重要性 (Feature Importance)', fontsize=12)
plt.title(f'Top {top_n} 特征重要性 — 宠物领养预测', fontsize=14)
plt.gca().invert_yaxis()

# 添加数值标签
for i, (bar, val) in enumerate(zip(bars, top_features['importance'].values)):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# 打印排名
print("\n=== 特征重要性排名 (Top 10) ===")
top_features[['feature', 'importance']].head(10)


## 10. 总结

### 关键发现

1. **五分类 vs 二分类**: 将 AdoptionSpeed 从 5 类简化为"已领养/未领养"二分类后，模型预测更稳定，准确率显著提升。

2. **ROC-AUC 验证**: 通过 ROC 曲线和 AUC 值验证了模型具有良好的区分能力，远优于随机猜测（AUC > 0.5）。

3. **特征重要性**: 年龄、品种、体型、照片数量等特征是影响领养速度的关键因素。

### 后续优化方向

- 利用 `Description` 文本描述做 NLP 特征
- 尝试 XGBoost / LightGBM 等梯度提升模型
- 超参数调优 (GridSearchCV)
- 交叉验证更稳健地评估模型
